In [105]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
import ipympl
import matplotlib.pyplot as plt
%matplotlib ipympl
import seaborn as sns
import json
from pathlib import Path
import ncaa_bbStats as nb
import ncaa_bbStats.player_utils as pu
import os
import re

In [129]:
# Assuming it's a CSV; change to pd.read_json if it's a JSON file
mapping_path = Path("/Users/averyhamel/Desktop/Syracuse/SAL 603/repositories/FinalProject/CollegeBaseballStatsPackage/src/data/team_names_stats/all_div_teams_standardized.csv")
team_map = pd.read_csv(mapping_path)

# Ensure the column names are clean
team_map.columns = team_map.columns.str.strip().str.lower()
# Standardize the names in the map for initial matching
team_map['match_key'] = team_map['team_name'].str.lower().str.strip()

display(team_map.head())

,team_id,team_name,division,match_key
0,1,A&M-Corpus Christi,1,a&m-corpus christi
1,2,AUM,2,aum
2,3,Abilene Christian University,1,abilene christian university
3,4,Abilene Christian,2,abilene christian
4,5,Academy of Art,2,academy of art


In [139]:
MY_DATA_PATH = Path("/Users/averyhamel/Desktop/Syracuse/SAL 603/repositories/FinalProject/CollegeBaseballStatsPackage/src/data/player_stats_cache")
master_csv_path = MY_DATA_PATH / "batting" / "batting_noMin.csv"

# Define column names
required_stats = [
    'g', 'ab', 'pa', 'h', '1b', '2b', '3b', 'hr', 'r', 'rbi', 
    'bb', 'so', 'hbp', 'sf', 'sh', 'gdp', 'sb', 'cs', 'avg', 
    'bb%', 'k%', 'bb/k', 'obp', 'slg', 'ops', 'iso', 'spd', 
    'babip', 'wsb', 'wrc', 'wraa', 'woba', 'wrc+'
]

# Define id columns
id_columns = ['year', 'name', 'team', 'team name'] 

try:
    # 1. Load data
    full_stats_df = pd.read_csv(master_csv_path, usecols=id_columns + required_stats)
    
    # 2. Filter for 2010-2025
    master_roster_df = full_stats_df[
        (full_stats_df['year'] >= 2010) & (full_stats_df['year'] <= 2025)
    ].copy()
    
    # 3. Clean strings
    master_roster_df['team name'] = master_roster_df['team name'].astype(str).str.strip()
    master_roster_df['name'] = master_roster_df['name'].astype(str).str.strip()

    # --- NEW: ATTACH STANDARDIZED ID AND DIVISION ---
    # Create the same match_key you used in the team_map
    master_roster_df['match_key'] = master_roster_df['team name'].str.lower().apply(
        lambda x: re.sub(r'[^\w\s]', '', x)
    )
    
    # Merge with team_map to pull in 'id' and 'division'
    # We only take the columns we need from team_map to keep it clean
    master_roster_df = pd.merge(
        master_roster_df, 
        team_map[['match_key', 'team_id', 'division']], 
        on='match_key', 
        how='left'
    )
    # ------------------------------------------------
    
    # 4. Sort by ID, player, season
    # Using 'id' here instead of 'team name' ensures consistency
    master_roster_df = master_roster_df.sort_values(['team_id', 'name', 'year'])
    
    print(f"Successfully imported and tagged {len(master_roster_df)} records.")
    print(master_roster_df[['year', 'name', 'team name', 'team_id', 'division']].head())

except FileNotFoundError:
    print(f"Error: Could not find the file at {master_csv_path}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print(master_roster_df.columns)

Successfully imported and tagged 26826 records.
       year            name                     team name  team_id  division
10004  2022  Aaron Staehely  Abilene Christian University      3.0       1.0
20637  2024  Aaron Staehely  Abilene Christian University      3.0       1.0
6396   2022       Adam Byrd  Abilene Christian University      3.0       1.0
1248   2021  Alexei Cazarin  Abilene Christian University      3.0       1.0
6529   2022  Alexei Cazarin  Abilene Christian University      3.0       1.0
Index(['name', 'team', 'team name', 'year', 'g', 'ab', 'pa', 'h', '1b', '2b',
       '3b', 'hr', 'r', 'rbi', 'bb', 'so', 'hbp', 'sf', 'sh', 'gdp', 'sb',
       'cs', 'avg', 'bb%', 'k%', 'bb/k', 'obp', 'slg', 'ops', 'iso', 'spd',
       'babip', 'wsb', 'wrc', 'wraa', 'woba', 'wrc+', 'match_key', 'team_id',
       'division'],
      dtype='object')


In [140]:
PITCH_PATH = Path("/Users/averyhamel/Desktop/Syracuse/SAL 603/repositories/FinalProject/CollegeBaseballStatsPackage/src/data/player_stats_cache/pitching")
pitch_csv = PITCH_PATH / "pitching_noMin.csv"

# 2. Define ID columns (matching your batting script)
id_columns = ['year', 'name', 'team', 'team name'] 

# 3. Define the Pitching Stats columns
pitching_stats = [
    'age', 'playerid', 'w', 'l', 'era', 'g', 'gs', 'cg', 'sho', 'sv', 'ip', 'tbf',
    'h', 'r', 'er', 'hr', 'bb', 'hbp', 'wp', 'bk', 'so',
    'k/9', 'bb/9', 'k/bb', 'hr/9', 'k%', 'bb%', 'k-bb%', 
    'avg', 'whip', 'babip', 'lob%', 'fip', 'e-f'
]

try:
    # 4. Load data
    full_pitch_df = pd.read_csv(pitch_csv, usecols=id_columns + pitching_stats)
    
    # 5. Filter for 2010-2025
    master_pitching_df = full_pitch_df[
        (full_pitch_df['year'] >= 2010) & (full_pitch_df['year'] <= 2025)
    ].copy()
    
    # 6. Clean strings
    master_pitching_df['team name'] = master_pitching_df['team name'].astype(str).str.strip()
    master_pitching_df['name'] = master_pitching_df['name'].astype(str).str.strip()

    # --- NEW: ATTACH STANDARDIZED ID AND DIVISION ---
    # Apply the same logic used in the batting script
    master_pitching_df['match_key'] = master_pitching_df['team name'].str.lower().apply(
        lambda x: re.sub(r'[^\w\s]', '', x)
    )
    
    # Merge with team_map (Dimension Table) to pull in 'team_id' and 'division'
    master_pitching_df = pd.merge(
        master_pitching_df, 
        team_map[['match_key', 'team_id', 'division']], 
        on='match_key', 
        how='left'
    )
    # ------------------------------------------------

    # 7. Data Cleaning: Ensure numeric types for modeling
    master_pitching_df['ip'] = pd.to_numeric(master_pitching_df['ip'], errors='coerce')
    master_pitching_df['era'] = pd.to_numeric(master_pitching_df['era'], errors='coerce')
    
    # 8. Sort by team_id, player, season
    master_pitching_df = master_pitching_df.sort_values(['team_id', 'name', 'year'])
    
    print(f"Successfully imported and tagged {len(master_pitching_df)} pitching records.")
    print(master_pitching_df[['year', 'name', 'team name', 'team_id', 'division']].head())

except FileNotFoundError:
    print(f"Error: Could not find the file at {pitch_csv}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print(master_pitching_df.columns)

Successfully imported and tagged 25964 pitching records.
       year             name                     team name  team_id  division
5997   2022        Adam Byrd  Abilene Christian University      3.0       1.0
10920  2023        Adam Byrd  Abilene Christian University      3.0       1.0
4354   2021  Adam Stephenson  Abilene Christian University      3.0       1.0
9424   2022  Adam Stephenson  Abilene Christian University      3.0       1.0
14483  2023  Adam Stephenson  Abilene Christian University      3.0       1.0
Index(['name', 'team', 'team name', 'age', 'playerid', 'year', 'w', 'l', 'era',
       'g', 'gs', 'cg', 'sho', 'sv', 'ip', 'tbf', 'h', 'r', 'er', 'hr', 'bb',
       'hbp', 'wp', 'bk', 'so', 'k/9', 'bb/9', 'k/bb', 'hr/9', 'k%', 'bb%',
       'k-bb%', 'avg', 'whip', 'babip', 'lob%', 'fip', 'e-f', 'match_key',
       'team_id', 'division'],
      dtype='object')


In [144]:
# --- STEP 1: PREPARE THE AGGREGATION MAPS ---
# We use 'first' for the string columns so they are preserved after the groupby
bat_cols_to_use = {k: v for k, v in batting_agg_map.items() if k in master_roster_df.columns}
pit_cols_to_use = {k: v for k, v in pitching_agg_map.items() if k in master_pitching_df.columns}

# --- STEP 2: AGGREGATE TO TEAM-SEASON LEVEL ---
# We group by the IDs, but "pull along" the team names using 'first'
print("Aggregating batting stats...")
team_batting_agg = master_roster_df.groupby(['team_id', 'year']).agg({
    **bat_cols_to_use,
    'team': 'first',
    'team name': 'first'
}).reset_index()

print("Aggregating pitching stats...")
team_pitching_agg = master_pitching_df.groupby(['team_id', 'year']).agg({
    **pit_cols_to_use,
    'team': 'first',
    'team name': 'first'
}).reset_index()

# --- STEP 3: JOIN THE TABLES BY ID ---
# We join on team_id and year. 
# Columns that exist in both (like 'team name') will get _bat and _pit suffixes.
master_performance_df = pd.merge(
    team_batting_agg, 
    team_pitching_agg, 
    on=['team_id', 'year'], 
    how='outer', 
    suffixes=('_bat', '_pit')
)

# --- STEP 4: CLEAN UP NAME COLUMNS ---
# Since we want one primary "team" and "team name" column, we fill any 
# missing batting names with the pitching names (and vice versa).
master_performance_df['team'] = master_performance_df['team_bat'].fillna(master_performance_df['team_pit'])
master_performance_df['team name'] = master_performance_df['team name_bat'].fillna(master_performance_df['team name_pit'])

# Drop the extra suffixed columns to keep the DF tidy
master_performance_df = master_performance_df.drop(columns=['team_bat', 'team_pit', 'team name_bat', 'team name_pit'])

# --- STEP 5: FINAL SORT ---
master_performance_df = master_performance_df.sort_values(['team_id', 'year'])

print(f"Join complete! 'team' and 'team name' are preserved, joined by 'team_id'.")
display(master_performance_df[['year', 'team_id', 'team', 'team name', 'ops', 'era']].head())

Aggregating batting stats...
Aggregating pitching stats...
Join complete! 'team' and 'team name' are preserved, joined by 'team_id'.


,year,team_id,team,team name,ops,era
0,2021,3.0,ACU,Abilene Christian University,0.763216,6.790354
1,2022,3.0,ACU,Abilene Christian University,0.738650,8.715737
2,2023,3.0,ACU,Abilene Christian University,0.696046,10.978145
3,2024,3.0,ACU,Abilene Christian University,0.689831,11.885957
4,2025,3.0,ACU,Abilene Christian University,0.722116,8.572085


In [145]:
# 1. Path Configuration
TEAM_CACHE_PATH = Path("/Users/averyhamel/Desktop/Syracuse/SAL 603/repositories/FinalProject/CollegeBaseballStatsPackage/src/data/team_stats_cache")

all_record_data = []

# 2. Aggressive Normalizer (to ensure match_key consistency)
def aggressive_normalize(name):
    if not isinstance(name, str): return ""
    name = name.lower()
    name = re.sub(r'\(.*?\)', '', name) # Remove league abbreviations in parentheses
    name = re.sub(r'[^a-z0-9]', '', name) # Remove all non-alphanumeric characters
    if name.endswith('st'):
        name = name[:-2] + 'state'
    return name.strip()

# 3. Load the JSON Standings (2010-2025)
for div in [1, 2, 3]:
    div_folder = TEAM_CACHE_PATH / f"div{div}"
    for year in range(2010, 2026):
        file_path = div_folder / f"{year}.json"
        
        if file_path.exists():
            with open(file_path, 'r') as f:
                data = json.load(f)
                
                # Filter out metadata keys like 'division' to keep only team dictionaries
                team_only_data = {k: v for k, v in data.items() if isinstance(v, dict)}
                
                # Create DataFrame
                df_year = pd.DataFrame.from_dict(team_only_data, orient='index')
                
                # Extract clean team name from the internal 'team' attribute
                if 'team' in df_year.columns:
                    df_year['team name'] = df_year['team']
                else:
                    # Fallback: Split the key at the first parenthesis
                    df_year['team name'] = df_year.index.str.split(' \(').str[0]
                
                # Keep only desired columns (W, L, T)
                # Note: Using .reindex ensures code doesn't crash if 'T' is missing in some files
                df_year = df_year.reindex(columns=['team name', 'W', 'L', 'T'])
                
                # Add Metadata
                df_year['year'] = year
                
                # Ensure numeric types
                for col in ['W', 'L', 'T']:
                    df_year[col] = pd.to_numeric(df_year[col], errors='coerce').fillna(0)
                
                # Create W_Pct variable
                # Formula: Wins / Total Games
                df_year['W_Pct'] = df_year['W'] / (df_year['W'] + df_year['L'] + df_year['T'])
                # Handle division by zero
                df_year['W_Pct'] = df_year['W_Pct'].fillna(0)
                
                # Prevent duplicate columns before appending
                df_year = df_year.loc[:, ~df_year.columns.duplicated()]
                
                all_record_data.append(df_year)

# 4. Combine into Master Records DataFrame
master_records_df = pd.concat(all_record_data, ignore_index=True)

# 5. Attach Standardized team_id and division
# Ensure mapping match_key is consistent
team_map['match_key'] = team_map['team_name'].apply(aggressive_normalize)
master_records_df['match_key'] = master_records_df['team name'].apply(aggressive_normalize)

master_records_df = pd.merge(
    master_records_df, 
    team_map[['match_key', 'team_id', 'division']], 
    on='match_key', 
    how='left'
)

# 6. Final Clean and Sort
master_records_df = master_records_df.dropna(subset=['team_id'])
master_records_df = master_records_df.sort_values(['team_id', 'year'])

print(f"Import complete! Created 'master_records_df' with {len(master_records_df)} records.")
display(master_records_df[['year', 'team name', 'team_id', 'division', 'W', 'L', 'T', 'W_Pct']].head())

Import complete! Created 'master_records_df' with 11225 records.


,year,team name,team_id,division,W,L,T,W_Pct
171,2010,A&M-Corpus Christi,1.0,1.0,20,33,1,0.370370
363,2011,A&M-Corpus Christi,1.0,1.0,37,24,0,0.606557
808,2012,A&M-Corpus Christi,1.0,1.0,24,33,0,0.421053
984,2013,A&M-Corpus Christi,1.0,1.0,33,24,0,0.578947
1329,2014,A&M-Corpus Christi,1.0,1.0,31,27,0,0.534483
